In [ ]:
import sys
import os
import time
import yaml
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
import pybullet as p

# ==========================================
# CONFIGURE WHICH STAGE TO WATCH TRUE LIVE:
STAGE_TO_WATCH = 0
# ==========================================

print(f"TRUE LIVE Tracker initialized. Monitoring current raw state of Stage {STAGE_TO_WATCH}...")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

env = RoomDroneEnv(gui=True, 
                   num_obstacles=NUM_OBS, 
                   randomize_obstacles=RAND_OBS, 
                   randomize_coins=RAND_COINS, 
                   reward_weights=reward_weights)

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'latest_model'))
print(f"Waiting for latest live brain at: {model_path}.zip")

while not os.path.exists(f"{model_path}.zip"):
    print("Waiting for the SaveLatestCallback to dump the first live memory...")
    time.sleep(5)

model = PPO.load(model_path, env=env)
obs, info = env.reset()

p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] TRUE LIVE Simulation started. You are watching the raw training state...")

while True:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action) 
    time.sleep(1./240.) 
    
    if terminated or truncated:
        time.sleep(0.5) 
        try:
            model = PPO.load(model_path, env=env)
        except Exception:
            pass 
            
        obs, info = env.reset()

: 